# ⚡ vLLM Colab 部署 + Prefix Cache 實測

> **目標**:在 Colab T4 GPU 上跑通 vLLM server,實測 prefix caching 對 long system prompt 的加速效果
>
> **環境**:Colab T4 GPU(免費 tier 可用)
>
> **Model**:Qwen2.5-0.5B-Instruct(494M params、FP16 ~1GB)
>
> **預期時間**:vLLM 啟動 2-4 min;benchmark 5-10 min

## 對應的 deep-dive

- 部署完整指南:[`../vLLM_部署實戰.md`](../vLLM_部署實戰.md)
- 引擎對比:[`../SGLang_TensorRT-LLM_對比.md`](../SGLang_TensorRT-LLM_對比.md)
- Speculative decoding:[`../../2.文字生成與解碼策略/EAGLE3_Speculative_Decoding_實作.md`](../../2.文字生成與解碼策略/EAGLE3_Speculative_Decoding_實作.md)
- 系統設計案例:[Case_02 LLM Gateway](../../../9.面試準備與職業發展/2.系統設計案例/Case_02_LLM_Gateway_API_Platform.md)

## phantom-mesh 寫由

在 phantom-mesh 中,vLLM 是處理「自托管模型」流量的核心。本 notebook 示範三個工程實戰要點:
1. **OpenAI-compatible API**:provider abstraction 的基礎(同一 client 可打 OpenAI / Anthropic / 自家 vLLM)
2. **prefix cache 命中率**:長 system prompt 場景的 cost killer(可降 70-90% TTFT)
3. **streaming SSE**:每個 provider stream 格式不同,vLLM 走 OpenAI 標準,phantom-mesh 統一 parser

---

## 0️⃣ 環境檢查

確認 T4 GPU 可用。Free Colab 的 T4 有 16GB VRAM,足夠跑 0.5B-1.5B model。

In [ ]:
import subprocess, sys, os

print('=== GPU ===')
out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv']).decode()
print(out)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()},Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
assert torch.cuda.is_available(), '❌ 需要 GPU runtime,請 Runtime → Change runtime type → T4 GPU'

## 1️⃣ 安裝 vLLM + 依賴

用 vLLM 0.6+(2026-05 stable)。安裝 ~3 min。

In [ ]:
%%capture
!pip install -U "vllm>=0.6.4" "openai>=1.50" "rich"

In [ ]:
import subprocess, time, requests, json
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
console = Console()

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
PORT = 8000
print('✅ Imports OK')

## 2️⃣ 啟動 vLLM Server(背景 process)

**重要參數**:
- `--gpu-memory-utilization 0.5` — Colab T4 16GB,留一半給其他(預設 0.9 會 OOM)
- `--max-model-len 4096` — 不要設太大,KV cache 會吃 VRAM
- `--enable-prefix-caching` — **本 notebook 重點**:開啟 automatic prefix caching
- `--dtype float16` — T4 不支援 bf16

啟動需要 2-4 分鐘(下載 model 1.2GB + load + compile CUDA graphs)。

In [ ]:
# 確保沒有殘留 process
subprocess.run(['pkill', '-f', 'vllm'], capture_output=True)
time.sleep(2)

vllm_cmd = [
    'vllm', 'serve', MODEL,
    '--port', str(PORT),
    '--gpu-memory-utilization', '0.5',
    '--max-model-len', '4096',
    '--max-num-seqs', '32',
    '--dtype', 'float16',
    '--enable-prefix-caching',  # 重點!
    '--disable-log-requests',
]

print('啟動 vLLM(背景 process)...')
log_file = open('/tmp/vllm.log', 'w')
proc = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=subprocess.STDOUT)
print(f'PID: {proc.pid}')
print('等 server 起來(~2-4 分鐘)...')

In [ ]:
# Poll health 直到 ready
MAX_WAIT = 300  # 5 min
t0 = time.time()
while time.time() - t0 < MAX_WAIT:
    try:
        r = requests.get(f'http://localhost:{PORT}/v1/models', timeout=2)
        if r.status_code == 200:
            models = r.json()['data']
            elapsed = time.time() - t0
            console.print(Panel(f'✅ vLLM ready in {elapsed:.0f}s\nModel: {models[0]["id"]}', border_style='green'))
            break
    except Exception:
        pass
    if proc.poll() is not None:
        print('❌ vLLM process exited unexpectedly. Tail of log:')
        os.system('tail -20 /tmp/vllm.log')
        raise RuntimeError('vLLM startup failed')
    time.sleep(5)
    print('.', end='', flush=True)
else:
    print('❌ Timeout. Tail of log:')
    os.system('tail -20 /tmp/vllm.log')
    raise RuntimeError('vLLM startup timeout')

## 3️⃣ 用 OpenAI Client 連 vLLM(OpenAI-compatible API)

vLLM 的 endpoint 跟 OpenAI 完全相容,**換 base_url 即可**。這是 phantom-mesh 等 gateway 喜歡的設計。

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=f'http://localhost:{PORT}/v1',
    api_key='dummy',  # vLLM 不檢查 key
)

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': '用 30 字介紹 LoRA 微調是什麼。'}],
    max_tokens=80,
)
console.print(Panel(resp.choices[0].message.content, title='✅ 第一個回應', border_style='green'))
print(f'Usage: prompt={resp.usage.prompt_tokens}, completion={resp.usage.completion_tokens}')

## 4️⃣ Streaming SSE 範例

vLLM 的 streaming 完全是 OpenAI SSE 格式。phantom-mesh stream parser 統一處理這格式 + Anthropic / Gemini 差異。

In [ ]:
import sys
print('Streaming output:\n', flush=True)
t0 = time.time()
first_token_time = None
total_tokens = 0
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': '用條列方式列出 Transformer 的 3 個重要組件。'}],
    max_tokens=120,
    stream=True,
)
for chunk in stream:
    if chunk.choices[0].delta.content:
        if first_token_time is None:
            first_token_time = time.time() - t0
        sys.stdout.write(chunk.choices[0].delta.content)
        sys.stdout.flush()
        total_tokens += 1
elapsed = time.time() - t0
print(f'\n\n📊 TTFT: {first_token_time*1000:.0f} ms | 總時間: {elapsed:.2f}s | tok/s: {total_tokens/elapsed:.1f}')

## 5️⃣ Prefix Cache 實測

**核心場景**:RAG / Agent 系統常有「長 system prompt + 短 user query」。
若 prefix cache 命中,vLLM 跳過 prefill 整段 system prompt,只 prefill user query → **TTFT 大幅下降**。

我們做 A/B:
- **Round 1**:5 個獨立 query,**首次跑**,prefix cache 都 miss
- **Round 2**:**同樣 5 個 query 再跑一次**,prefix cache 應該全 hit

In [ ]:
# 構造 ~2K token 的長 system prompt
LONG_SYSTEM_PROMPT = '\n'.join([
    '你是 phantom-mesh 的技術助理。請以繁體中文,專業但易懂地回答問題。',
    '回答原則:',
    '1. 先給結論,後展開細節',
    '2. 引用具體技術或數字',
    '3. 不確定的內容明確標註',
    '4. 避免廢話,3-5 點條列即可',
    '',
    '相關背景知識:',
    '- Transformer 由 Vaswani et al. 2017 提出,核心是 Self-Attention',
    '- Multi-Head Attention 把 attention 切多個 head 平行運算',
    '- BERT(Google 2018)是 encoder-only,用於理解任務',
    '- GPT(OpenAI 2018+)是 decoder-only,用於生成任務',
    '- T5(Google 2019)是 encoder-decoder',
    '- Flash Attention(Tri Dao 2022)解決 O(N²) memory 問題',
    '- LoRA(Hu et al. 2021)是 PEFT 主流方法',
    '- QLoRA(Dettmers 2023)結合 4-bit 量化 + LoRA',
    '- DPO(Rafailov 2023)繞過 reward model 直接優化偏好',
    '- vLLM(Kwon et al. 2023)用 PagedAttention 大幅提升 throughput',
    '- GraphRAG(Microsoft 2024)用社群偵測 + 多層摘要做主題綜合查詢',
    '- DeepSeek-R1(2025/01)純 RL + GRPO 誘發 reasoning',
    '- Llama 4 Scout(2025/04)用 iRoPE 從 256K 訓練外推到 10M context',
    '- EAGLE-3(2025)是當前最快的 speculative decoding 變體',
    '- C2PA 是 AI 內容真實性的開放標準',
    '- MCP(Anthropic 2024/11)是 agent-tool 通訊協定',
    '- A2A(Google 2025)是 agent-agent 協定',
] * 3)

print(f'System prompt 長度:約 {len(LONG_SYSTEM_PROMPT)} chars / 估計 ~{len(LONG_SYSTEM_PROMPT)//3} tokens')

QUERIES = [
    'LoRA 跟 full fine-tune 有什麼差別?',
    'Flash Attention 為何能加速?',
    'GraphRAG 跟 vector RAG 的差別?',
    'DPO 為何能繞過 reward model?',
    'vLLM 的 PagedAttention 原理?',
]

In [ ]:
def measure(query, system_prompt):
    t0 = time.time()
    first_token_t = None
    completion_tokens = 0
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': query},
        ],
        max_tokens=80,
        stream=True,
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            if first_token_t is None:
                first_token_t = time.time() - t0
            completion_tokens += 1
    total_t = time.time() - t0
    return first_token_t, total_t, completion_tokens

# Round 1 — cold cache(首次)
print('=== Round 1(cold cache, prefix cache miss)===')
round1 = []
for q in QUERIES:
    ttft, total, tokens = measure(q, LONG_SYSTEM_PROMPT)
    round1.append((ttft, total, tokens))
    print(f'  TTFT {ttft*1000:6.0f} ms | total {total:.2f}s | tokens {tokens:3d} | Q: {q[:30]}...')

# Round 2 — warm cache(重跑)
print('\n=== Round 2(warm cache, prefix cache hit)===')
round2 = []
for q in QUERIES:
    ttft, total, tokens = measure(q, LONG_SYSTEM_PROMPT)
    round2.append((ttft, total, tokens))
    print(f'  TTFT {ttft*1000:6.0f} ms | total {total:.2f}s | tokens {tokens:3d} | Q: {q[:30]}...')

## 6️⃣ 對比表 — Prefix Cache 加速幅度

In [ ]:
table = Table(title='Prefix Cache 對 TTFT 的影響')
table.add_column('Query', style='cyan')
table.add_column('Cold TTFT (ms)', justify='right')
table.add_column('Warm TTFT (ms)', justify='right')
table.add_column('Speedup', justify='right', style='green')
for q, (c, _, _), (w, _, _) in zip(QUERIES, round1, round2):
    speedup = c / w if w > 0 else 0
    table.add_row(q[:25] + '...', f'{c*1000:.0f}', f'{w*1000:.0f}', f'{speedup:.1f}x')

avg_cold = sum(r[0] for r in round1) / len(round1) * 1000
avg_warm = sum(r[0] for r in round2) / len(round2) * 1000
table.add_row('—— 平均 ——', f'{avg_cold:.0f}', f'{avg_warm:.0f}', f'{avg_cold/avg_warm:.1f}x', style='bold')
console.print(table)

print(f'\n📊 觀察:warm cache TTFT 大約是 cold 的 1/{avg_cold/avg_warm:.1f}')
print('   實際生產:RAG / Agent 高重複 system prompt 場景,prefix cache 是 cost killer'
      '\n   參考:Anthropic prompt caching 報導 75-90% 成本節省')

## 7️⃣ Throughput Benchmark — Continuous Batching

vLLM 的 continuous batching 允許多個 request 共享 batch。
我們同時送 10 個 request(用 asyncio),看吞吐量。

In [ ]:
import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(base_url=f'http://localhost:{PORT}/v1', api_key='dummy')

async def one_req(q):
    t0 = time.time()
    r = await async_client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': q}],
        max_tokens=80,
    )
    return time.time() - t0, r.usage.completion_tokens

async def benchmark():
    queries = [f'用 30 字介紹 {topic}' for topic in [
        'Transformer', 'LoRA', 'GraphRAG', 'DPO', 'vLLM', 'Flash Attention', 'MCP', 'RAG', 'CoT', 'PEFT',
    ]]
    t0 = time.time()
    results = await asyncio.gather(*[one_req(q) for q in queries])
    total_t = time.time() - t0
    total_tok = sum(r[1] for r in results)
    return total_t, total_tok, results

total_t, total_tok, results = await benchmark()
console.print(Panel(
    f'10 個 concurrent requests 完成\n'
    f'總時間:{total_t:.2f}s\n'
    f'總 tokens:{total_tok}\n'
    f'**Aggregate throughput:{total_tok/total_t:.0f} tok/s**\n'
    f'(單請求 throughput 約 {sum(r[1] for r in results)/sum(r[0] for r in results):.0f} tok/s,\n'
    f' aggregate 提升 ~{(total_tok/total_t) / (sum(r[1] for r in results)/sum(r[0] for r in results)):.1f}x 證明 continuous batching 在做事)',
    title='📊 Continuous Batching Throughput', border_style='green'
))

## 8️⃣ phantom-mesh 真實工程考量

### 8.1 Prefix Cache 命中率監控
- vLLM Prometheus metrics 有 `vllm:cache_hit_ratio`,生產要接 Grafana dashboard
- RAG / Agent 場景目標 hit rate > 50%;若 < 30% 表示 system prompt 變動太頻繁
- 對應 [`../vLLM_部署實戰.md` §6](../vLLM_部署實戰.md)

### 8.2 GPU Memory Utilization 調整
- `--gpu-memory-utilization 0.9` 預設(production)
- 留多少給 KV cache:`(VRAM × 0.9 - model_size) / KV per token / seq_len`
- Long context 場景 KV cache 比 model weight 還大

### 8.3 Quantization 部署
- 想跑更大 model:`--quantization awq` 或 `--quantization fp8`
- T4 不支援 FP8;A100/H100 上才能用
- 對應 [`../../5.監督微調 (SFT)/Quantization_Primer.md`](../../5.監督微調%20(SFT)/Quantization_Primer.md)

### 8.4 Speculative Decoding
- vLLM 已內建 EAGLE-3,啟動加 `--speculative-config '{"model":"<draft_model>","num_speculative_tokens":5}'`
- 預期 2.5-6× 加速,本 notebook 沒測是因為 0.5B 模型已經很快、speculation overhead 反而傷
- 對應 [`../../2.文字生成與解碼策略/EAGLE3_Speculative_Decoding_實作.md`](../../2.文字生成與解碼策略/EAGLE3_Speculative_Decoding_實作.md)

### 8.5 Multi-tenant LoRA Serving
- `--enable-lora --max-loras 8` 啟動多 adapter 服務
- Request header 帶 `lora-adapter-name`,vLLM 自動 swap
- 對應 Notebook 1 訓出的 adapter 可在這裡部署

### 8.6 Provider Fallback
- 自家 vLLM 掛掉 → fallback 到 OpenAI / Anthropic
- phantom-mesh provider abstraction 的核心,只要 base_url + api_key 一換就行
- 對應 [Case_02 LLM Gateway §5.7](../../../9.面試準備與職業發展/2.系統設計案例/Case_02_LLM_Gateway_API_Platform.md)

### 8.7 Streaming SSE 統一
- vLLM 走 OpenAI SSE 格式(本 notebook 第 4 cell 測過)
- Anthropic、Gemini 各家格式不同
- phantom-mesh stream_parser 對齊各家 → 對外一個介面

---

## 9️⃣ 收尾 — 關閉 vLLM

釋放 GPU 記憶體。

In [ ]:
import psutil

try:
    parent = psutil.Process(proc.pid)
    for child in parent.children(recursive=True):
        child.terminate()
    parent.terminate()
    print('✅ vLLM stopped')
except Exception as e:
    subprocess.run(['pkill', '-f', 'vllm'], capture_output=True)
    print(f'(用 pkill 強制關閉,reason: {e})')

time.sleep(3)
import torch
torch.cuda.empty_cache()
print(f'GPU mem 釋放後: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 🔬 擴展練習

1. **跑更大 model**:換成 `Qwen/Qwen2.5-1.5B-Instruct` 或 `Qwen/Qwen2.5-3B-Instruct`(後者需 `--quantization awq`)
2. **接 LoRA**:用 Notebook 1 訓的 adapter,加 `--enable-lora --lora-modules my=./adapter` 啟動
3. **加 EAGLE-3 speculative**:用 0.5B 當 draft + 3B target,測加速比
4. **接 Langfuse 監控**:每個 request 自動 trace、cost + token + latency
5. **多 GPU(本地)**:`--tensor-parallel-size 2` 在多卡機器跑 70B model
6. **跑 SGLang 對比**:同 model 用 SGLang(`pip install sglang`),比 prefix cache 命中率與 throughput
7. **Disaggregated Prefill**:vLLM 支援 P/D 分離,生產 RAG 場景可省 50% TTFT
8. **Tool Calling**:vLLM 支援 `tool_choice='auto'`,搭 LangGraph(Notebook 2)做完整 agent stack

---

## 📚 References

- [vLLM 官方文檔](https://docs.vllm.ai/)
- [vLLM Prefix Caching](https://docs.vllm.ai/en/stable/design/prefix_caching.html)
- [vLLM Speculative Decoding](https://docs.vllm.ai/en/latest/features/speculative_decoding/)
- 本 repo:[`../vLLM_部署實戰.md`](../vLLM_部署實戰.md)、[`../SGLang_TensorRT-LLM_對比.md`](../SGLang_TensorRT-LLM_對比.md)、[`../../2.文字生成與解碼策略/EAGLE3_Speculative_Decoding_實作.md`](../../2.文字生成與解碼策略/EAGLE3_Speculative_Decoding_實作.md)

---

**Last updated**: 2026-05-16  
**Tested on**: Colab T4 (Free tier),Python 3.10,vllm 0.6.4,openai 1.50,Qwen2.5-0.5B-Instruct